# Avaliação: Assistente de IA Generativa com Gemini

**Disciplina:** Disruptive Architectures: IoT, IoB & Generative AI

**Grupo:**
- Diego Andrade dos Santos (RM566385)
- Grazielle de Alencar Silva (RM561529)
- Vinicius Soteras Braga (RM566230)

**Tema escolhido:** 4. Assistente de saúde e bem-estar (IoB)

> Observação: o Hugging Face retornou `402 Payment Required` (créditos gratuitos esgotados), por isso todas as etapas foram feitas com o Gemini.

Os trechos marcados com **`# >>> PERSONALIZE`** foram adaptados ao tema do grupo.

## 0. Configuração

In [5]:
!pip install google-genai gradio -q

In [6]:
from google import genai
from google.genai import types
from google.colab import userdata

In [7]:
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
print("GEMINI_API_KEY ok" if GEMINI_API_KEY else "ERRO: adicione GEMINI_API_KEY nos Secrets do Colab")

GEMINI_API_KEY ok


In [8]:
MODELO_GEMINI = "gemini-3.5-flash-lite"
cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)

In [9]:
# >>> PERSONALIZE: system do assistente de saúde e bem-estar
SYSTEM_PROMPT = """Você é um assistente de saúde e bem-estar.
Ajude o usuário a entender dados de wearables (passos, sono e frequência cardíaca) e dê dicas gerais de hábitos saudáveis.
Responda sempre em português, de forma clara, em no máximo 5 frases.
Nunca dê diagnóstico médico nem indique remédios. Se os dados parecerem fora do normal, recomende procurar um profissional de saúde.
Se a pergunta não tiver relação com saúde e bem-estar, diga educadamente que não pode ajudar."""

In [10]:
def perguntar_gemini(pergunta, system, temperatura=0.7):
    """Envia uma pergunta ao Gemini e devolve o texto da resposta."""
    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=pergunta,
        config=types.GenerateContentConfig(
            system_instruction=system,
            temperature=temperatura,
            max_output_tokens=300,
        ),
    )
    return resposta.text

---
## Etapa 1: Assistente com guardrails

A mesma pergunta é enviada com três `system` diferentes: sem restrição, com restrição de tema e com restrição de tema e formato.

In [11]:
# >>> PERSONALIZE: três system para o tema do grupo
systems = {
    "Sem restrição": "Você é um assistente de saúde e bem-estar. Responda em português.",
    "Restrição de tema": SYSTEM_PROMPT,
    "Tema + formato": SYSTEM_PROMPT + "\nResponda sempre em no máximo 3 tópicos curtos e termine com: 'Isto não substitui uma avaliação médica.'",
}

# >>> PERSONALIZE: uma pergunta dentro do escopo e uma fora
perguntas = [
    "Dormi 5 horas e meu smartwatch marcou 72 bpm em repouso. O que esses dados indicam?",
    "Qual a melhor configuração de computador para jogos?",
]

for pergunta in perguntas:
    print("PERGUNTA:", pergunta)
    for nome, system in systems.items():
        print(f"\n===== {nome} =====")
        print(perguntar_gemini(pergunta, system))
    print("\n" + "=" * 60 + "\n")

PERGUNTA: Dormi 5 horas e meu smartwatch marcou 72 bpm em repouso. O que esses dados indicam?

===== Sem restrição =====
Olá! Como seu assistente de saúde e bem-estar, estou aqui para ajudar a interpretar esses dados, mas lembre-se de que as informações a seguir têm caráter educativo e não substituem a avaliação de um profissional médico.

Aqui está uma análise do que esses dois dados — **5 horas de sono** e **72 bpm de frequência cardíaca em repouso (FCR)** — podem indicar:

### 1. As 5 horas de sono (Privação de sono)
* **O que indica:** É menos do que a média recomendada para adultos (que geralmente é de 7 a 9 horas por noite). 
* **Impacto imediato:** Dormir apenas 5 horas pode causar fadiga diurna, diminuição da concentração, irritabilidade e afeta o tempo de reação. 
* **Impacto a longo prazo:** A privação crônica de sono está associada a um sistema imunológico enfraquecido, maior risco de ganho de peso, estresse elevado e problemas cardiovasculares.

### 2. Os 72 bpm em repouso


**Observações do grupo (Etapa 1):**

- **Com o guardrail de tema, o assistente recusou a pergunta fora do escopo?**
  Sim. Sem restrição, o assistente chegou a falar sobre a configuração de computador, porque o `system` só definia o papel, sem proibir outros assuntos. Com a restrição de tema, ele respondeu educadamente que só pode ajudar com saúde e bem-estar.

- **Com a restrição de formato, o estilo da resposta mudou?**
  Sim. A resposta ficou mais curta e organizada em tópicos, e terminou com o aviso de que não substitui uma avaliação médica. Na pergunta dentro do escopo, as versões com restrição também evitaram dar diagnóstico e sugeriram procurar um profissional, o que é essencial para o nosso tema.

---
## Etapa 2: Comparação com temperatura

Cada pergunta é enviada com `temperature=0.0` e `temperature=1.0`.

In [12]:
# >>> PERSONALIZE: 2 perguntas sobre o tema do grupo
perguntas = [
    "Minha média de passos caiu de 9.000 para 4.000 por dia. Como posso me movimentar mais?",
    "O que significa a variabilidade da frequência cardíaca que meu smartwatch mostra?",
]

for p in perguntas:
    print("PERGUNTA:", p)
    for temp in [0.0, 1.0]:
        print(f"\n--- Gemini | temperature={temp} ---")
        print(perguntar_gemini(p, SYSTEM_PROMPT, temperatura=temp))
    print("\n" + "=" * 60 + "\n")

PERGUNTA: Minha média de passos caiu de 9.000 para 4.000 por dia. Como posso me movimentar mais?

--- Gemini | temperature=0.0 ---
Para aumentar seus passos novamente, tente incorporar pequenas caminhadas ao longo do dia, como usar as escadas ou dar uma volta rápida no quarteirão. Outra boa estratégia é estabelecer metas progressivas, começando com 5.000 passos e subindo aos poucos. Caminhar ouvindo um podcast ou convidar um amigo pode tornar o exercício mais divertido e constante. Lembre-se de que o importante é a consistência, e não voltar ao ritmo anterior do dia para a noite. Se sentir qualquer desconforto físico persistente ao se movimentar, consulte um profissional de saúde.

--- Gemini | temperature=1.0 ---
Para recuperar o ritmo, tente adicionar pequenas caminhadas de 10 minutos após as refeições ou use a escada em vez do elevador. 
Estabeleça metas realistas, como 5.000 passos nos primeiros dias, e vá aumentando gradualmente. 
Caminhar ouvindo um podcast ou convidar um amigo p

**Observações do grupo (Etapa 2):**

- **Com `temperature=0.0` as respostas ficaram mais consistentes?**
  Sim. Com 0.0 o modelo escolhe sempre as palavras mais prováveis, então as respostas são mais diretas e, ao executar de novo, saem praticamente iguais.

- **Com `temperature=1.0` ficaram mais variadas?**
  Sim. As respostas mudaram a cada execução, com outras palavras e outras sugestões (por exemplo, dicas diferentes para aumentar os passos).

- **Qual combinação é mais adequada para o tema?**
  Para um assistente de saúde, a temperatura baixa (perto de 0.0) é mais adequada, porque explicações sobre dados de saúde precisam ser consistentes e seguras. A temperatura alta pode ser útil só para pedidos criativos, como ideias de atividades. Não foi possível comparar com o Hugging Face porque a conta retornou erro 402 (créditos esgotados).

---
## Etapa 3: Chat com memória

Quem guarda a conversa é o nosso código, na lista `historico`, que é enviada inteira ao Gemini a cada mensagem.

Comandos: `sair` | `limpar` | `historico`

In [13]:
historico = []

print("Chat iniciado! Comandos: sair | limpar | historico\n")

while True:
    entrada = input("Você: ")

    if entrada.lower() == "sair":
        print("Encerrando chat.")
        break

    if entrada.lower() == "limpar":
        historico = []
        print("\n(histórico apagado)\n")
        continue

    if entrada.lower() == "historico":
        print(f"\n(mensagens no histórico: {len(historico)})\n")
        continue

    historico.append({"role": "user", "parts": [{"text": entrada}]})

    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=historico,
        config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, max_output_tokens=300),
    )
    texto = resposta.text

    historico.append({"role": "model", "parts": [{"text": texto}]})
    print(f"\nAssistente: {texto}\n")

Chat iniciado! Comandos: sair | limpar | historico



KeyboardInterrupt: Interrupted by user

**Observações do grupo (Etapa 3):**

- **O assistente lembrou do histórico? Por quê?**
  Sim. Depois de contar o nome e um dado do smartwatch, o assistente respondeu corretamente quando perguntamos de novo. Depois do comando `limpar`, ele não sabia mais. Isso acontece porque o modelo não tem memória própria: cada chamada é independente. Quem guarda a conversa é o nosso código, na lista `historico`, que é enviada inteira a cada mensagem. Por esse mesmo motivo, se trocássemos de modelo mantendo a lista, o novo modelo também "lembraria" da conversa.

---
## Etapa 4: Interface web com Gradio

Interface com slider de temperatura.

In [14]:
import gradio as gr

def texto_da_mensagem(conteudo):
    """Extrai o texto de uma mensagem do histórico do Gradio."""
    if isinstance(conteudo, str):
        return conteudo
    if isinstance(conteudo, list):
        return " ".join(item.get("text", "") for item in conteudo if isinstance(item, dict))
    return str(conteudo)


def responder(mensagem, historico_gradio, temperatura):
    conteudos = []
    for msg in historico_gradio:
        papel = "model" if msg["role"] == "assistant" else "user"
        conteudos.append({"role": papel, "parts": [{"text": texto_da_mensagem(msg["content"])}]})
    conteudos.append({"role": "user", "parts": [{"text": mensagem}]})

    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=conteudos,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            temperature=temperatura,
            max_output_tokens=300,
        ),
    )
    return resposta.text

In [15]:
# >>> PERSONALIZE: título, descrição e exemplos de acordo com o tema do grupo.
gr.ChatInterface(
    fn=responder,
    additional_inputs=[gr.Slider(0.0, 1.0, value=0.7, step=0.1, label="Temperatura")],
    title="Assistente de Saúde e Bem-estar",
    description="Pergunte sobre os dados do seu wearable: passos, sono e frequência cardíaca.",
    examples=[
        ["Como melhorar a qualidade do meu sono?", 0.0],
        ["Me dê ideias para andar mais durante o dia", 1.0],
    ],
    cache_examples=False,
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://31e1708ff259ebeb62.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Observações do grupo (Etapa 4):**

- **Com temperatura baixa as respostas ficaram mais previsíveis? Com temperatura alta ficaram mais criativas?**
  Sim. Com o slider em 0.0, a mesma pergunta gerou respostas praticamente iguais e mais objetivas. Com 1.0, as respostas variaram mais e trouxeram sugestões diferentes, principalmente em pedidos de ideias (como formas de andar mais). Mesmo com temperatura alta, o assistente continuou respeitando o `system` e não deu diagnóstico.

> Para parar a interface, interrompa a célula no Colab.

---
## Etapa 5 (Bônus): API com sessões independentes

Cada `session_id` tem seu próprio histórico no servidor.

In [16]:
%%writefile app_avaliacao.py
import os
from fastapi import FastAPI
from pydantic import BaseModel
from google import genai
from google.genai import types

cliente = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODELO = "gemini-3.5-flash-lite"
SYSTEM_PROMPT = os.environ.get("SYSTEM_PROMPT", "Você é um assistente prestativo.")

app = FastAPI()
sessoes = {}  # session_id -> histórico daquela sessão

class Pergunta(BaseModel):
    session_id: str
    mensagem: str

@app.post("/chat")
def chat(pergunta: Pergunta):
    historico = sessoes.setdefault(pergunta.session_id, [])
    historico.append({"role": "user", "parts": [{"text": pergunta.mensagem}]})

    resposta = cliente.models.generate_content(
        model=MODELO,
        contents=historico,
        config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, max_output_tokens=300),
    )
    historico.append({"role": "model", "parts": [{"text": resposta.text}]})
    return {"resposta": resposta.text}

Writing app_avaliacao.py


In [17]:
# Inicia o servidor em segundo plano
!pip install fastapi uvicorn -q

import os, subprocess, time
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["SYSTEM_PROMPT"] = SYSTEM_PROMPT

servidor = subprocess.Popen(["uvicorn", "app_avaliacao:app", "--port", "8000"])
time.sleep(5)
print("Servidor rodando em http://localhost:8000")

Servidor rodando em http://localhost:8000


In [18]:
# Sessão A sabe o nome, sessão B não sabe
import requests

def enviar(sessao, mensagem):
    r = requests.post("http://localhost:8000/chat", json={"session_id": sessao, "mensagem": mensagem})
    print(f"[{sessao}] {mensagem}\n-> {r.json()['resposta']}\n")

enviar("A", "Meu nome é Ana e hoje dei 6.000 passos.")
enviar("A", "Qual é o meu nome?")
enviar("B", "Qual é o meu nome?")

[A] Meu nome é Ana e hoje dei 6.000 passos.
-> Olá, Ana! 6.000 passos é uma marca excelente e representa um ótimo movimento para o seu dia. Tente manter uma rotina de caminhadas regulares para cuidar da saúde do coração e do corpo. Lembre-se também de beber bastante água e manter uma boa qualidade de sono. Se notar qualquer alteração incomum no seu corpo, procure sempre um profissional de saúde.

[A] Qual é o meu nome?
-> Seu nome é Ana. Como posso te ajudar com seus dados de saúde e bem-estar hoje?

[B] Qual é o meu nome?
-> Como sou um assistente focado em saúde e bem-estar, não tenho acesso a informações pessoais como o seu nome. Posso ajudar você a analisar seus dados de wearables, como passos, sono e frequência cardíaca, além de dar dicas de hábitos saudáveis. Se precisar de orientações sobre esses temas, estou à disposição!



In [19]:
# Encerra o servidor
servidor.terminate()
print("Servidor encerrado.")

Servidor encerrado.


**Observações do grupo (Etapa 5):**

- **A sessão A respondeu o nome corretamente? A sessão B não sabia?**
  Sim. A sessão A respondeu "Ana", porque o nome estava no histórico dela. A sessão B disse que não sabia, porque o histórico dela começou vazio.

- **O que diferencia essa API da atividade anterior?**
  Na atividade anterior, a API não guardava nada: cada requisição tinha só o `system` e a pergunta atual. Agora o servidor guarda um histórico separado para cada `session_id`, então a API tem memória por usuário e várias pessoas podem conversar ao mesmo tempo sem misturar as conversas.